In [1]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import uproot as up
import pickle
import nue_booster
import importlib
importlib.reload(nue_booster)

import awkward

In [2]:
# SURPRISE SAMPLES
u_nu4a = up.open("/exp/uboone/data/users/kpletcher/slimmedFiles/slimmedMCC910/run4a/nuepresel/checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_4a.root")["nuselection"]["SubRun"]
u_nu4a_nufilter = up.open("/exp/uboone/data/users/kpletcher/slimmedFiles/slimmedMCC910/run4a/nuepresel/checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_4a.root")["nuselection"]["NeutrinoSelectionFilter"]

u_nue4a = up.open("/exp/uboone/data/users/kpletcher/slimmedFiles/slimmedMCC910/run4a/nuepresel/checkout_MCC9.10_Run4a4c4d5_v10_04_07_13_BNB_intrinsic_nue_overlay_surprise_reco2_hist_4a.root")["nuselection"]["SubRun"]
u_nue4a_nufilter = up.open("/exp/uboone/data/users/kpletcher/slimmedFiles/slimmedMCC910/run4a/nuepresel/checkout_MCC9.10_Run4a4c4d5_v10_04_07_13_BNB_intrinsic_nue_overlay_surprise_reco2_hist_4a.root")["nuselection"]["NeutrinoSelectionFilter"]
# u_dirt4a = up.open("/exp/uboone/data/users/kpletcher/slimmedFiles/slimmedMCC910/run4a/nuepresel/checkout_MCC9.10_Run4a4c4d5_v10_04_07_13_BNB_dirt_overlay_surprise_reco2_hist_4a.root")["nuselection"]["SubRun"]
# u_ncpi04a = up.open("/exp/uboone/data/users/kpletcher/slimmedFiles/slimmedMCC910/run4a/nuepresel/checkout_MCC9.10_Run4a_v10_04_07_16_BNB_NCpi0_overlay_surprise_reco2_hist.root")["nuselection"]["SubRun"]
# u_ext4a = up.open("/exp/uboone/data/users/kpletcher/slimmedFiles/slimmedMCC910/run4a/nuepresel/checkout_MCC9.10_Run4a_BNB_beam_off_data_surprise_reco2_hist.root")["nuselection"]["SubRun"]

In [11]:
pd.set_option('display.max_rows', None)

vars = ["pot", "run", "subRun"]
vars_query = ["run", "sub", "nu_pdg", "ccnc", "mcf_pass_ncpi0", "category"]

nu4a_subRun = u_nu4a.arrays(vars, library='pd')
nu4a_nueSelFilter = u_nu4a_nufilter.arrays(vars_query, library='pd')
nu4a_duplicates = pd.merge(nu4a_subRun, nu4a_nueSelFilter, left_on=["run","subRun"], right_on=["run","sub"], how="left")
nu4a = nu4a_duplicates.drop_duplicates(subset=['subRun','pot'])

nue4a_subRun = u_nue4a.arrays(vars, library='pd')
nue4a_nueSelFilter = u_nue4a_nufilter.arrays(vars_query, library='pd')
nue4a_duplicates = pd.merge(nue4a_subRun, nue4a_nueSelFilter, left_on=["run","subRun"], right_on=["run","sub"], how="left")
nue4a = nue4a_duplicates.drop_duplicates(subset=['subRun','pot'])

# print(nu4a_subRun)
# print(nu4a_nueSelFilter)
# print(nu4a)

# print(nue4a_subRun)
# print(nue4a_nueSelFilter)
print(nue4a)
# print(nue4a_drops)

NUEQUERY = '(abs(nu_pdg)==12 & ccnc==0)'
NCPI0QUERY = '(mcf_pass_ncpi0==1)'
MCQUERY = '((abs(nu_pdg)==12 & ccnc==0) | mcf_pass_ncpi0==1)'

nu4a_ccnue = nu4a.query(NUEQUERY)
nu4a_ncpi0 = nu4a.query(NCPI0QUERY)
nu4a_no_ccnue_ncpi0 = nu4a.query('~'+MCQUERY)

train_nu4a, test_nu4a = train_test_split(nu4a_no_ccnue_ncpi0, test_size=0.5, random_state=1990)

ZPQUERY = 'category==10'
NPQUERY = 'category==11'
NUEOTHERQUERY = '(category!=10 & category!=11)'

nue4a_zp = nue4a.query(ZPQUERY)
nue4a_np = nue4a.query(NPQUERY)
nue4a_other = nue4a.query(NUEOTHERQUERY)

signal_nue4a = nue4a_zp
bkg_nue4a = pd.concat([nue4a_np, nue4a_other], ignore_index=True)

train_signal_nue4a, test_signal_nue4a = train_test_split(signal_nue4a, test_size=0.5)
train_bkg_nue4a, test_bkg_nue4a = train_test_split(bkg_nue4a, test_size=0.5)

test_nue4a = pd.concat([test_signal_nue4a, test_bkg_nue4a], ignore_index=True)
train_nue4a = pd.concat([train_signal_nue4a, train_bkg_nue4a], ignore_index=True)

# NUESIGNALQUERY = 'category==10'
# NUEBKGQUERY = 'category!=10'

# signal_nue4a = nue4a.query(NUESIGNALQUERY)
# bkg_nue4a = nue4a.query(NUEBKGQUERY)

# train_signal_nue4a, test_signal_nue4a = train_test_split(signal_nue4a, test_size=0.5)
# train_bkg_nue4a, test_bkg_nue4a = train_test_split(bkg_nue4a, test_size=0.5)

# test_nue4a = pd.concat([test_signal_nue4a, test_bkg_nue4a], ignore_index=True)
# train_nue4a = pd.concat([train_signal_nue4a, train_bkg_nue4a], ignore_index=True)

                pot    run  subRun    sub  nu_pdg  ccnc  mcf_pass_ncpi0  \
0      8.817092e+18  18961      65   65.0    12.0   0.0             0.0   
4      5.398213e+18  18961      66   66.0    12.0   0.0             0.0   
5      3.897486e+18  18961      70    NaN     NaN   NaN             NaN   
6      1.276157e+19  18961      71   71.0    12.0   0.0             0.0   
11     3.268107e+18  18961      72   72.0    12.0   0.0             0.0   
15     1.088795e+19  18961      77   77.0    12.0   0.0             0.0   
18     7.497466e+18  18961      79   79.0    12.0   0.0             0.0   
25     5.847978e+18  18961      82   82.0    12.0   0.0             0.0   
30     4.697070e+18  18961     100  100.0    12.0   0.0             0.0   
38     5.374742e+18  18961     103  103.0    12.0   0.0             0.0   
44     5.868704e+18  18961     104  104.0    12.0   0.0             0.0   
47     8.307965e+18  18961     126  126.0    12.0   0.0             0.0   
53     1.011909e+19  1896

In [16]:
# Run 4a Test POT
train_nu4a_pot = 0
test_nu4a_pot = 0
nu4a_ccnue_pot = 0
nu4a_ncpi0_pot = 0
nu4a_no_ccnue_ncpi0_pot = 0

test_signal_nue4a_pot = 0
train_signal_nue4a_pot = 0
test_bkg_nue4a_pot = 0
train_bkg_nue4a_pot = 0

signal_nue4a_pot = 0
bkg_nue4a_pot = 0

train_nue4a_pot = 0
test_nue4a_pot = 0

nue4a_pot = 0
nu4a_pot = 0

nue4a_subRun_pot = 0

for i, row in train_nu4a.iterrows():
    train_nu4a_pot+=row["pot"]

for i, row in test_nu4a.iterrows():
    test_nu4a_pot+=row["pot"]

for i, row in nu4a_ccnue.iterrows():
    nu4a_ccnue_pot+=row["pot"]

for i, row in nu4a_ncpi0.iterrows():
    nu4a_ncpi0_pot+=row["pot"]

for i, row in nu4a_no_ccnue_ncpi0.iterrows():
    nu4a_no_ccnue_ncpi0_pot+=row["pot"]

for i, row in test_signal_nue4a.iterrows():
    test_signal_nue4a_pot+=row["pot"]

for i, row in train_signal_nue4a.iterrows():
    train_signal_nue4a_pot+=row["pot"]

for i, row in test_bkg_nue4a.iterrows():
    test_bkg_nue4a_pot+=row["pot"]

for i, row in train_bkg_nue4a.iterrows():
    train_bkg_nue4a_pot+=row["pot"]

for i, row in signal_nue4a.iterrows():
    signal_nue4a_pot+=row["pot"]

for i, row in bkg_nue4a.iterrows():
    bkg_nue4a_pot+=row["pot"]

for i, row in nue4a.iterrows():
    nue4a_pot+=row["pot"]

for i, row in test_nue4a.iterrows():
    test_nue4a_pot+=row["pot"]

for i, row in train_nue4a.iterrows():
    train_nue4a_pot+=row["pot"]

for i, row in nu4a.iterrows():
    nu4a_pot+=row["pot"]

for i, row in nue4a_subRun.iterrows():
    nue4a_subRun_pot+=row["pot"]


In [17]:
print("Nu 4a Train Sample =", train_nu4a_pot)
print("Nu 4a Test Sample =", test_nu4a_pot)
print("Nu 4a CC Nue Events =", nu4a_ccnue_pot)
print("Nu 4a NC Pi0 Events =", nu4a_ncpi0_pot)
print("Nu 4a Non CC Nue and NC Pi0 Events =", nu4a_no_ccnue_ncpi0_pot)
print()
print("Nue 4a Signal Train Sample (combining signal and background) =", train_signal_nue4a_pot)
print("Nue 4a Signal Test Sample (combining signal and background) =", test_signal_nue4a_pot)
print("Nue 4a Background Train Sample (combining signal and background) =", train_bkg_nue4a_pot)
print("Nue 4a Background Test Sample (combining signal and background) =", test_bkg_nue4a_pot)
print()
print("Nue Signal Sample POT =", signal_nue4a_pot)
print("Nue Background Sample POT =", bkg_nue4a_pot)
print()
print("Nue Overlay POT, just after merging =", nue4a_pot)
print("Nu Overlay POT, just after merging =", nu4a_pot)

print("Nue Overlay SubRun Branch POT =", nue4a_subRun_pot)

Nu 4a Train Sample = 1.1148815987443866e+20
Nu 4a Test Sample = 1.1203258623742444e+20
Nu 4a CC Nue Events = 2.1712261555400212e+18
Nu 4a NC Pi0 Events = 8.883990361412731e+18
Nu 4a Non CC Nue and NC Pi0 Events = 2.235207461118631e+20

Nue 4a Signal Train Sample (combining signal and background) = 2.44083003121722e+21
Nue 4a Signal Test Sample (combining signal and background) = 2.361477544070097e+21
Nue 4a Background Train Sample (combining signal and background) = 1.5877470755720947e+22
Nue 4a Background Test Sample (combining signal and background) = 1.6123545718662355e+22

Nue Signal Sample POT = 4.802307575287317e+21
Nue Background Sample POT = 3.20010164743833e+22

Nue Overlay POT, just after merging = 3.680332404967062e+22
Nu Overlay POT, just after merging = 2.3457596262881585e+20
Nue Overlay SubRun Branch POT = 3.680332404967062e+22


In [18]:
print("Total Run 4a POT, No CC Nue or NC Pi0 =", train_nu4a_pot+test_nu4a_pot)
print("Total Run 4a Nu Overlay POT =", nu4a_ccnue_pot+nu4a_ncpi0_pot+nu4a_no_ccnue_ncpi0_pot)
print()
print("Total Run 4a Nue Overlay POT =", train_nue4a_pot+test_nue4a_pot)

Total Run 4a POT, No CC Nue or NC Pi0 = 2.235207461118631e+20
Total Run 4a Nu Overlay POT = 2.3457596262881585e+20

Total Run 4a Nue Overlay POT = 3.680332404967062e+22
